In [1]:
import numpy                    as np
import pandas                   as pd
import matplotlib.pyplot        as plt
import seaborn                  as sns

from matplotlib.ticker          import StrMethodFormatter

import re


# --- Hugging Face Datasets
from datasets import load_dataset, DatasetDict
from datasets.features import ClassLabel



URL = "https://huggingface.co/datasets/khalidalt/HuffPost/resolve/main/News_Category_Dataset_v2.json"
huff_all = load_dataset("json", data_files=URL, split="train")

df = huff_all.to_pandas()


-----------------

# take the author info

In [2]:
def pull_author_specialty(author, marker: list):
    if not isinstance(author, str):
        return False

    # author_lower = author.lower()
    # return any(
    #     specialty in author_lower
    #     for specialty in marker
    # )
    ## __Match the first marker, ignoring capitalization
    pattern = "|".join(re.escape(item) for item in marker if item)
    if not pattern:
        return pd.NA

    match = re.search(pattern, author, flags=re.IGNORECASE)
    if match is None:
        return pd.NA

    extra_words_to_replace = ['editor', 'author']

    pattern_after = r"\b(?:" + "|".join( re.escape(word) for word in extra_words_to_replace) + r")\b"

    ## __Extract text after the marker and clean newline characters
    specialty = (
        author[match.end():]
        .replace("\\n", " ")
        .strip(" \t\r\n,:;-")
    )

    specialty = re.sub(pattern_after, "", specialty, flags=re.IGNORECASE) 
    ## __Normalize spaces and newlines after removal
    specialty = " ".join(specialty.split()).strip(",:;- ")



    return specialty if specialty else pd.NA




possible_markers_for_specialty = ['contributor', 'author', 'specialist']

df_author = (
    df.sample(10)
    .assign( info = lambda x: x['authors'].apply(pull_author_specialty, marker = possible_markers_for_specialty))

)


df_author

,category,headline,authors,link,short_description,date,info
94991,POLITICS,Indiana Takes on America: Discrimination Again...,"Richard Brodsky, ContributorSenior Fellow, Demos",https://www.huffingtonpost.com/entry/indiana-t...,"Americans now understand what the Indiana ""Rel...",2015-03-29,"Senior Fellow, Demos"
60778,PARENTS,How To Make Child Care Stop Costing As Much As...,,https://www.huffingtonpost.comhttp://pubx.co/o...,Soaring college costs are a hot topic on the p...,2016-04-21,NaN
152120,FOOD & DRINK,14 Great Grilling Recipes for the 4th of July,"Jennifer Segal, Contributor\nChef, Cookbook Au...",https://www.huffingtonpost.com/entry/fourth-of...,Forget the standard fare. Throw the ultimate J...,2013-07-02,"Chef, Cookbook , Busy Mom"
144836,PARENTING,Early Education: Playing to Win,"Helen Blank, Contributor\nDirector of Child Ca...",https://www.huffingtonpost.com/entry/early-edu...,Several U.S. Senators and Representatives join...,2013-09-18,Director of Child Care and Early Learning at t...
181843,DIVORCE,Divorced TV Characters: Our 15 Favorite Divorc...,Brittany Wong,https://www.huffingtonpost.com/entry/divorced-...,Turn on your television now and you're bound t...,2012-08-20,NaN
187369,STYLE & BEAUTY,Yves Saint Laurent Name Change: No More 'Yves'...,,https://www.huffingtonpost.com/entry/yves-sain...,"According to Women's Wear Daily, Hedi Slimane ...",2012-06-21,NaN
36932,ENTERTAINMENT,"'Teen Wolf' Fans, You'll Never Guess What Stil...",Minou Clark,https://www.huffingtonpost.com/entry/teen-wolf...,We didn't see that coming.,2017-01-18,NaN
176491,WELLNESS,To Honor,"Mark Nepo, Contributor\nAuthor, New York Times...",https://www.huffingtonpost.com/entry/poetry-so...,"Once we know something, experience something, ...",2012-10-16,New York Times bestseller ‘The Book of Awakening’
127228,STYLE & BEAUTY,Telling the Stories of SXSW One Outfit At a Time,"Elisa Goodkind and Lily Mandelbaum, Contributo...",https://www.huffingtonpost.com/entry/the-story...,Charli XCX told us all about how Baby Spice an...,2014-03-24,www.stylelikeu.com
11505,MEDIA,My Story: TED And Sexual Harassment,"Jordan Reeves, ContributorFounder, VideoOut",https://www.huffingtonpost.com/entry/my-story-...,It’s no surprise that so many have been sharin...,2017-11-17,"Founder, VideoOut"


In [3]:
# def view_content(df):
#     with pd.option_context("display.max_colwidth", None):
#         display(df)

from IPython.display import display, HTML

def view_content(df):
    if isinstance(df, pd.Series):
        df = df.to_frame()

    with pd.option_context("display.max_colwidth", None):
        html = df.to_html(
            max_rows=None,
            max_cols=None,
            escape=True,
        )

    display(HTML(
        '<div style="width:100%; overflow-x:auto;">'
        '<style>'
        '.full-text-table td, .full-text-table th {'
        '    white-space:pre-wrap;'
        '    overflow-wrap:anywhere;'
        '    min-width:100px;'
        '    max-width:500px;'
        '    text-align:left;'
        '    vertical-align:top;'
        '}'
        '</style>'
        + html.replace('class="dataframe"', 'class="dataframe full-text-table"')
        + '</div>'
    ))

In [4]:
test = df_author.query("info.notna()")[['authors','category', 'info']]
view_content(test)

,authors,category,info
94991,"Richard Brodsky, ContributorSenior Fellow, Demos",POLITICS,"Senior Fellow, Demos"
152120,"Jennifer Segal, Contributor\nChef, Cookbook Author, Busy Mom",FOOD & DRINK,"Chef, Cookbook , Busy Mom"
144836,"Helen Blank, Contributor\nDirector of Child Care and Early Learning at the National Wome...",PARENTING,Director of Child Care and Early Learning at the National Wome...
176491,"Mark Nepo, Contributor\nAuthor, New York Times bestseller ‘The Book of Awakening’",WELLNESS,New York Times bestseller ‘The Book of Awakening’
127228,"Elisa Goodkind and Lily Mandelbaum, Contributor\nwww.stylelikeu.com",STYLE & BEAUTY,www.stylelikeu.com
11505,"Jordan Reeves, ContributorFounder, VideoOut",MEDIA,"Founder, VideoOut"


-------

# merging labels

In [ ]:
label_combo = [
    'WORLDPOST','WORLD NEWS' ,'THE WORLDPOST'
    'PARENTS', 'PARENTING'
    'STYLE', 'STYLE & BEAUTY'
    "TASTE"  ,   "FOOD & DRINK",
    # "DIVORCE", "WEDDINGS"
    "GREEN", "ENVIRONMENT"
    # "MONEY" , "BUSINESS"
    "HEALTHY LIVING" , "WELLNESS"
    # "SCIENCE", "TECH"
    # "COLLEGE", "EDUCATION"
]

In [ ]:

## __Define proposed groups
label_groups = {
    'ARTS & CULTURE'        :   [ 'ARTS', 'ARTS & CULTURE', 'CULTURE & ARTS'],
    'WORLD NEWS'            :   [ 'WORLDPOST','WORLD NEWS' ,'THE WORLDPOST'   ],
    'PARENTING'             :   [ 'PARENTS', 'PARENTING'   ],
    'STYLE & BEAUTY'        :   [ 'STYLE', 'STYLE & BEAUTY'   ],
    'FOOD & DRINK'          :   [ 'TASTE'  , 'FOOD & DRINK',   ],
    'ENVIRONMENT'           :   [ 'GREEN', 'ENVIRONMENT'   ],
    'WELLNESS'              :   [ 'HEALTHY LIVING' , 'WELLNESS'   ],  
}

## __Convert grouped labels into a lookup dictionary
label_mapping = {
    original: merged
    for merged, originals in label_groups.items()
    for original in originals
}

df_topics = (
    df
    .assign(
        label1_clean = lambda x: (
            x['label1']
            .astype('string')
            .str.lower()
            .str.strip()
            .replace(r'\s*&\s*', '&', regex=True)
        ),
        month = lambda x: (
            pd.to_datetime(x['date'], errors='coerce')
            .dt.to_period('M')
        ),
    )
    .assign(
        label2 = lambda x: (
            x['label1_clean']
            .map(label_mapping)
            .fillna(x['label1_clean'])
        )
    )
)

---

# end